# Notebook 3 — Forensic Analysis + Gradio App

**Run this notebook from:** `notebooks\`

**Requires at least one of these files in** `notebooks\data\processed\`:
- `phase3_with_severity.csv` — from Notebook 1
- `phase4_meme_analysis.csv` — from Notebook 2

**What this notebook does:**
1. Combines both pipeline outputs into one dataset
2. Generates severity distribution charts
3. Generates a PDF forensic report
4. Launches a Gradio web app — upload any severity CSV to get an instant report

---
Run cells **one at a time**. Each cell prints `✅` when it succeeds.

In [1]:
# ── CELL 1 — Imports ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # prevents display errors in Jupyter on Windows
import matplotlib.pyplot as plt
import gradio as gr
import nest_asyncio
from pathlib import Path
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import pagesizes, colors
from reportlab.lib.units import inch

nest_asyncio.apply()  # fixes WinError 10054 Gradio crash in Jupyter on Windows

PROCESSED = Path('data') / 'processed'

print('✅ Cell 1 OK')
print(f'   PROCESSED : {PROCESSED.resolve()}')

✅ Cell 1 OK
   PROCESSED : C:\Users\tejas\Major_Project\AI_Digital_Evidence_Extraction\notebooks\data\processed


In [2]:
# ── CELL 2 — Load & Combine Pipeline Outputs ──────────────────────────────
dfs = []

text_file = PROCESSED / 'phase3_with_severity.csv'
meme_file = PROCESSED / 'phase4_meme_analysis.csv'

if text_file.exists():
    df_text = pd.read_csv(text_file)[['clean_text', 'severity_score']].copy()
    df_text['source'] = 'text'
    dfs.append(df_text)
    print(f'   Text pipeline : {len(df_text):,} rows')
else:
    print(f'   ⚠️  Not found : {text_file.name} — run Notebook 1 first')

if meme_file.exists():
    df_meme = pd.read_csv(meme_file)[['clean_text', 'severity_score']].copy()
    df_meme['source'] = 'meme'
    dfs.append(df_meme)
    print(f'   Meme pipeline : {len(df_meme):,} rows')
else:
    print(f'   ⚠️  Not found : {meme_file.name} — run Notebook 2 first')

assert len(dfs) > 0, (
    '\n❌ No input files found.'
    '\n   Run Notebook 1 → phase3_with_severity.csv'
    '\n   Run Notebook 2 → phase4_meme_analysis.csv'
)

df_combined = pd.concat(dfs, ignore_index=True)
df_combined['severity_score'] = (
    df_combined['severity_score'].fillna(0).astype(int)
)

out = PROCESSED / 'phase5_combined.csv'
df_combined.to_csv(out, index=False)

print(f'\n✅ Cell 2 OK — {len(df_combined):,} total rows combined')
print('\nSeverity breakdown:')
print(
    df_combined['severity_score'].value_counts().sort_index()
    .rename({0: '0 — Low', 1: '1 — Moderate', 2: '2 — High', 3: '3 — Critical'})
)

   Text pipeline : 47,215 rows
   Meme pipeline : 8,500 rows

✅ Cell 2 OK — 55,715 total rows combined

Severity breakdown:
severity_score
0 — Low         30009
1 — Moderate     5068
2 — High         5182
3 — Critical    15456
Name: count, dtype: int64


In [3]:
# ── CELL 3 — Severity Distribution Charts ─────────────────────────────────
counts     = df_combined['severity_score'].value_counts().sort_index()
labels     = ['Low', 'Moderate', 'High', 'Critical']
values     = [int(counts.get(i, 0)) for i in range(4)]
clr        = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
axes[0].pie(
    values, labels=labels, autopct='%1.1f%%',
    colors=clr, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Risk Distribution — Pie', fontsize=13, fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, values, color=clr, edgecolor='black', width=0.5)
axes[1].set_title('Risk Distribution — Bar', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Record Count')
for bar, val in zip(bars, values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(values) * 0.01,
        f'{val:,}', ha='center', va='bottom', fontweight='bold'
    )

plt.tight_layout()
chart_path = PROCESSED / 'risk_distribution.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Cell 3 OK — chart saved to {chart_path.name}')


✅ Cell 3 OK — chart saved to risk_distribution.png


C:\Users\tejas\AppData\Local\Temp\ipykernel_12184\1517724256.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ── CELL 4 — PDF Report Generator ─────────────────────────────────────────
def generate_pdf_report(df, output_path='forensic_report.pdf'):
    """
    Generates a structured PDF forensic report.
    df must have a 'severity_score' column (0/1/2/3).
    Returns path string of the generated PDF.
    """
    total    = len(df)
    counts   = df['severity_score'].value_counts()
    low      = int(counts.get(0, 0))
    moderate = int(counts.get(1, 0))
    high     = int(counts.get(2, 0))
    critical = int(counts.get(3, 0))
    crit_pct = critical / total * 100 if total > 0 else 0
    high_pct = high     / total * 100 if total > 0 else 0

    if crit_pct > 5:
        risk_level = 'CRITICAL RISK EXPOSURE'
    elif high_pct > 10:
        risk_level = 'HIGH RISK EXPOSURE'
    else:
        risk_level = 'MODERATE RISK EXPOSURE'

    doc      = SimpleDocTemplate(str(output_path), pagesize=pagesizes.A4)
    styles   = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph(
        'AI-Based Digital Behavioral Forensic Report', styles['Heading1']
    ))
    elements.append(Spacer(1, 0.15 * inch))
    elements.append(Paragraph(
        'Project: AI-Driven Digital Behavioral Forensic Analysis System '
        'for Detecting Harmful Online Interactions',
        styles['Normal']
    ))
    elements.append(Spacer(1, 0.25 * inch))

    elements.append(Paragraph('Evidence Summary', styles['Heading2']))
    elements.append(Spacer(1, 0.1 * inch))

    table_data = [
        ['Risk Level',    'Count',          'Percentage'],
        ['Low (0)',       f'{low:,}',        f'{low/total*100:.1f}%'],
        ['Moderate (1)',  f'{moderate:,}',   f'{moderate/total*100:.1f}%'],
        ['High (2)',      f'{high:,}',       f'{high/total*100:.1f}%'],
        ['Critical (3)', f'{critical:,}',   f'{crit_pct:.1f}%'],
        ['TOTAL',        f'{total:,}',      '100.0%'],
    ]
    table = Table(table_data, colWidths=[3 * inch, 1.5 * inch, 1.5 * inch])
    table.setStyle(TableStyle([
        ('BACKGROUND',     (0, 0),  (-1, 0),  colors.HexColor('#2c3e50')),
        ('TEXTCOLOR',      (0, 0),  (-1, 0),  colors.whitesmoke),
        ('FONTNAME',       (0, 0),  (-1, 0),  'Helvetica-Bold'),
        ('FONTNAME',       (0, 1),  (-1, -1), 'Helvetica'),
        ('ROWBACKGROUNDS', (0, 1),  (-1, -1),
         [colors.HexColor('#f2f2f2'), colors.white]),
        ('GRID',           (0, 0),  (-1, -1), 0.5, colors.black),
        ('ALIGN',          (1, 0),  (-1, -1), 'CENTER'),
        ('FONTNAME',       (0, -1), (-1, -1), 'Helvetica-Bold'),
    ]))
    elements.append(table)
    elements.append(Spacer(1, 0.3 * inch))

    elements.append(Paragraph(
        f'Overall Classification: {risk_level}', styles['Heading2']
    ))
    elements.append(Spacer(1, 0.15 * inch))
    elements.append(Paragraph(
        'This automated system analyzes digital evidence from social media '
        'and messaging platforms. The severity scoring logic identifies '
        'behavioral risk indicators that may contribute to psychological '
        'distress, cyberbullying, or online harassment. Results should be '
        'reviewed by a qualified digital forensic investigator before use '
        'in any formal context.',
        styles['Normal']
    ))

    doc.build(elements)
    return str(output_path)

# Test with combined data
test_pdf = generate_pdf_report(
    df_combined, PROCESSED / 'forensic_report.pdf'
)
print(f'✅ Cell 4 OK — PDF generated: {test_pdf}')

✅ Cell 4 OK — PDF generated: data\processed\forensic_report.pdf


In [5]:
# ── CELL 5 — Gradio Analysis Function ─────────────────────────────────────
def analyze_file(file):
    """
    Gradio handler.
    Input : uploaded CSV with 'severity_score' column
    Output: summary text, matplotlib chart, PDF path
    """
    if file is None:
        return 'Please upload a CSV file.', None, None

    try:
        df = pd.read_csv(file.name)
    except Exception as e:
        return f'❌ Could not read file: {e}', None, None

    if 'severity_score' not in df.columns:
        return (
            "❌ CSV must contain a 'severity_score' column\n"
            "   Valid values: 0=Low  1=Moderate  2=High  3=Critical\n"
            f"   Columns found: {list(df.columns)}",
            None, None
        )

    df['severity_score'] = (
        pd.to_numeric(df['severity_score'], errors='coerce')
        .fillna(0).astype(int)
    )

    total    = len(df)
    counts   = df['severity_score'].value_counts()
    low      = int(counts.get(0, 0))
    moderate = int(counts.get(1, 0))
    high     = int(counts.get(2, 0))
    critical = int(counts.get(3, 0))
    crit_pct = critical / total * 100 if total > 0 else 0
    high_pct = high     / total * 100 if total > 0 else 0

    if crit_pct > 5:
        risk_meter = '🔴 CRITICAL RISK DETECTED'
    elif high_pct > 10:
        risk_meter = '🟠 HIGH RISK DETECTED'
    else:
        risk_meter = '🟡 MODERATE RISK DETECTED'

    summary = (
        f'📊 Total Records Analyzed : {total:,}\n\n'
        f'🟢 Low Risk      : {low:,}  ({low/total*100:.1f}%)\n'
        f'🟡 Moderate Risk : {moderate:,}  ({moderate/total*100:.1f}%)\n'
        f'🟠 High Risk     : {high:,}  ({high/total*100:.1f}%)\n'
        f'🔴 Critical Risk : {critical:,}  ({crit_pct:.1f}%)\n\n'
        f'🚨 Overall Risk Classification:\n{risk_meter}'
    )

    # Chart
    clr    = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
    vals   = [low, moderate, high, critical]
    lbls   = ['Low', 'Moderate', 'High', 'Critical']
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        vals, labels=lbls, autopct='%1.1f%%',
        colors=clr, startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    ax.set_title('Risk Distribution', fontsize=13, fontweight='bold')
    plt.tight_layout()

    # PDF
    pdf_path = generate_pdf_report(df, 'forensic_report.pdf')

    return summary, fig, pdf_path

print('✅ Cell 5 OK — analyze_file() ready')

✅ Cell 5 OK — analyze_file() ready


In [6]:
# ── CELL 6 — Launch Gradio App ─────────────────────────────────────────────
#
# A public shareable link will print below (valid 72 hours)
#
# Upload any of these CSVs from notebooks\data\processed\
#   phase3_with_severity.csv   → text data
#   phase4_meme_analysis.csv   → meme data
#   phase5_combined.csv        → both combined

with gr.Blocks(title='AI Digital Forensic Analyzer',
               theme=gr.themes.Soft()) as demo:

    gr.Markdown('# 🧠 AI-Based Digital Behavioral Forensic Analysis System')
    gr.Markdown(
        'Upload a processed CSV with a `severity_score` column '
        '(0=Low, 1=Moderate, 2=High, 3=Critical) to generate an instant '
        'behavioral risk analysis report and downloadable PDF.'
    )

    with gr.Row():
        with gr.Column(scale=1):
            file_input  = gr.File(file_types=['.csv'], label='📁 Upload CSV')
            analyze_btn = gr.Button(
                '🔍 Analyze Evidence', variant='primary', size='lg'
            )
            pdf_out = gr.File(label='📄 Download Forensic PDF Report')
        with gr.Column(scale=2):
            summary_out = gr.Textbox(
                label='Forensic Summary', lines=12, interactive=False
            )
            plot_out = gr.Plot(label='Risk Distribution Chart')

    analyze_btn.click(
        fn=analyze_file,
        inputs=file_input,
        outputs=[summary_out, plot_out, pdf_out]
    )

demo.launch(share=True)

Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 4.19.1, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://965c9aafe7f825c1cc.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
